# # 종합 실습 가이드_Feature Engineering



- 실습 주제 : Titanic 생존 예측 모델 생성
- 실습 목표 : Feature Engineering 기법만을 이용하여 Test 데이터 기준 Accuracy를 최대화 하기
- 실습 조건
  - 아래 Baseline Code 를 참고하여 **(필수)** 과정은 제공된 코드 그대로 사용하고, 그외 EDA, Feature Transformation, Feature Creation, Feature Selection 과정은 자유롭게 선택
  - Target 변수 : Survived (생존 여부) *Target 변수는 어떠한 데이타 변환도 하지 않음
  - 파생 변수 생성 : 반드시 실습코드에 없는 파생변수를 최소 1개 이상 신규 생성해야 함
  - Data Partition : **아래 코드 그대로 사용**
  - 데이타 건수 : 데이타 제거는 하지 않음 (전체/Train/Test 건수 그대로 유지), 중복 데이타가 있어도 제거하지 않음
  - 학습 모델 및 모델 검증, 평가 방법 : **아래 코드 그대로 사용. 하이퍼파라미터 변경 등 모델 튜닝 작업은 허용 안됨**  
- 실습 제출물
   - 종합실습_결과물_Feature Engineering_OOO.xlsx 에 실습 결과 작성 후 코드 (.ipynb)와 같이 제출
   - 실습 결과는 최종 선정된 모델 기준으로 작성
   - 제출시 화일명은 본인 이름으로 2개 화일 제출 (엑셀, 코드)   
     예) **종합실습_결과물_Feature Engineering_홍길동.xlsx, 종합실습_결과물_Feature Engineering_홍길동.ipynb**

# Baaeline Code

데이타 불러오기 **(필수)**

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

불필요한 변수 제거 **(필수)**

In [ ]:
df = df.drop(columns=["PassengerId", "Ticket"])

Data Partition **(필수)**

In [ ]:
from sklearn.model_selection import train_test_split

# Data Partition
df_train, df_test = train_test_split(
    df,
    test_size=0.3,     # test 30%
    random_state=42,   # 재현성
    shuffle=True       # 섞어서 분리
)

One-Hot Encoding

In [ ]:
# 범주형 컬럼 선택
cat_cols = df_train.select_dtypes(include=['object', 'category']).columns.tolist()

# k-1 dummy 변수화
# Train Set
df_train = pd.get_dummies(df_train, columns=cat_cols, drop_first=True)   # True: k-1개 dummy, False: One-Hot Encoding

# Test Set
df_test = pd.get_dummies(df_test, columns=cat_cols, drop_first=True)

# test를 train 컬럼 구조에 맞추기 (없는 컬럼은 0으로, train에 없는 test컬럼은 제거)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

X, y 분리 **(필수)**

In [ ]:
# train
Y_train = df_train['Survived']
X_train = df_train.drop(columns=['Survived'])

# test
Y_test = df_test['Survived']
X_test = df_test.drop(columns=['Survived'])

모델 학습 및 평가 **(필수)**

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# 모델 학습 : RandomForestClassifier
rf_final = RandomForestClassifier(n_estimators=300, random_state=42)
rf_final.fit(X_train, Y_train)

# 모델 평가
acc_train = accuracy_score(Y_train, rf_final.predict(X_train))
acc_test  = accuracy_score(Y_test, rf_final.predict(X_test))

print(f"\n[최종 모델 성능]")
print(f"변수 수   : {len(X_train.columns)}")
print(f"Train 성능 : {acc_train:.4f}")
print(f"Test 성능  : {acc_test:.4f}")